# Exploring the construction permits for the Tableau test (layer 23)

The question: how many **business days** does it take to approve a new Public Space Construction Permit in fiscal year 2026 (1 October 2025 – 30 September 2026), overall, by permit type, and by another useful split?

There is no data dictionary, so before building anything I checked:

1. **Load:** the layer and its key columns
2. **Which permits:** new, issued in FY26, applied on or after 1 October 2025, one row per permit
3. **What "approve" means:** which date marks the approval
4. **Missing permits:** whether the layer has already dropped some FY26 permits
5. **Permit types:** how to group them into categories
6. **Other splits:** which segments show real differences

Each section ends with the decision it led to.

*This notebook uses the Python test's environment (`python_test/.venv`, kernel "CS12 Python Test (3.11)"), since it needs the same packages. The same layer was explored for the GIS test (renewal permits); here the focus is new permits.*

In [1]:
import re

import numpy as np
import pandas as pd
import requests

pd.set_option("display.width", 160)

PERMITS_23 = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_APPS/DDOT_TOPS/MapServer/23"
TIMEZONE = "America/New_York"
FY26_START = pd.Timestamp("2025-10-01", tz=TIMEZONE)
FY27_START = pd.Timestamp("2026-10-01", tz=TIMEZONE)

## 1. Load the data

The whole layer, page by page (the service hands out at most 2,000 rows per request).

In [2]:
# The layer's own description: column names, types and code lists
layer_info = requests.get(PERMITS_23, params={"f": "json"}, timeout=60).json()

# Ask for the rows page by page until the service says there are no more
rows, offset = [], 0
while True:
    params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "false",
        "orderByFields": "OBJECTID",  # same order every time, so pages don't overlap
        "resultOffset": offset,
        "resultRecordCount": 2000,
        "f": "json",
    }
    page = requests.get(f"{PERMITS_23}/query", params=params, timeout=120).json()
    if "error" in page:               # a bad query still comes back as "OK"
        raise RuntimeError(page["error"])
    rows += [feature["attributes"] for feature in page["features"]]
    if not page.get("exceededTransferLimit"):
        break
    offset += len(page["features"])
permits = pd.DataFrame(rows)

# Dates arrive as milliseconds since 1970 in UTC, so convert them to DC time
for field in layer_info["fields"]:
    if field["type"] == "esriFieldTypeDate":
        permits[field["name"]] = pd.to_datetime(permits[field["name"]], unit="ms", utc=True).dt.tz_convert(TIMEZONE)

print(f"{len(permits):,} rows, {permits['TrackingNumber'].nunique():,} permits, {permits.shape[1]} columns")
print("Date columns:", [f["name"] for f in layer_info["fields"] if f["type"] == "esriFieldTypeDate"])

55,332 rows, 10,142 permits, 100 columns
Date columns: ['IntakeDate', 'ApplicationDate', 'ReadyForReviewDate', 'AssignedDate', 'LastApprovalDate', 'IssueDate', 'EffectiveDate', 'ExpirationDate', 'PermitFeesPaymentDate', 'InspectionPaymentDate', 'MeterPaymentDate', 'LastUpdateDate', 'PCReviewStatusDate', 'LastResubmissionDate', 'LastResubmissionRequestDate', 'CellInstallationDate']


## 2. Which permits

The brief: application date on or after 1 October 2025, and only **new** permits that have been **issued in FY26**. From the GIS exploration, `Renewal = 1` means New, and a permit counts as issued when it has an issue date (it keeps it when its status later moves to "Permit Expired").

In [3]:
# Apply the brief's filters one at a time and count what's left
steps = {
    "All rows": pd.Series(True, index=permits.index),
    "New (Renewal = 1)": permits["Renewal"] == 1,
    "Applied on/after 1 Oct 2025": permits["ApplicationDate"] >= FY26_START,
    "Issued (has an issue date)": permits["IssueDate"].notna(),
    "Issued in FY26": (permits["IssueDate"] >= FY26_START) & (permits["IssueDate"] < FY27_START),
}
keep = pd.Series(True, index=permits.index)
counts = []
for step, rule in steps.items():
    keep &= rule
    counts.append({"filter": step, "rows": keep.sum(), "permits": permits.loc[keep, "TrackingNumber"].nunique()})
print(pd.DataFrame(counts).to_string(index=False))

fy26 = permits[keep]
print()
print(fy26["StatusDescription"].value_counts())

                     filter  rows  permits
                   All rows 55332    10142
          New (Renewal = 1) 27157     5536
Applied on/after 1 Oct 2025 22838     4981
 Issued (has an issue date) 13218     3061
             Issued in FY26 12155     2770

StatusDescription
Issued            10420
Permit Expired     1723
Suspended            11
Revoked               1
Name: count, dtype: int64


In [4]:
# A permit has one row per location. Are the permit-level columns the same on all its rows?
permit_columns = ["ApplicationDate", "IssueDate", "LastApprovalDate", "StatusDescription", "TypeDetailNames",
                  "IsExcavation", "IsPaving", "IsOccupancy", "ReviewProcess", "PermitTypeDescription",
                  "DaysToApprove", "DaysToIssue", "PermitteeName", "TotalFee"]
differs = {column: (fy26.groupby("TrackingNumber")[column].nunique(dropna=False) > 1).sum() for column in permit_columns}
print("Permits where a column differs between its rows:", differs)
print()
print("Rows per permit:", fy26.groupby("TrackingNumber").size().describe().round(1).to_dict())

# Permits applied in FY26 but issued after it ended
late = permits[(permits["Renewal"] == 1) & (permits["ApplicationDate"] >= FY26_START) & (permits["IssueDate"] >= FY27_START)]
print(f"Applied in FY26 but issued after 30 Sep 2026: {late['TrackingNumber'].nunique()} permits (excluded)")

Permits where a column differs between its rows: {'ApplicationDate': np.int64(0), 'IssueDate': np.int64(0), 'LastApprovalDate': np.int64(0), 'StatusDescription': np.int64(0), 'TypeDetailNames': np.int64(0), 'IsExcavation': np.int64(0), 'IsPaving': np.int64(0), 'IsOccupancy': np.int64(0), 'ReviewProcess': np.int64(0), 'PermitTypeDescription': np.int64(0), 'DaysToApprove': np.int64(0), 'DaysToIssue': np.int64(0), 'PermitteeName': np.int64(0), 'TotalFee': np.int64(0)}

Rows per permit: {'count': 2770.0, 'mean': 4.4, 'std': 7.8, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 2.0, 'max': 75.0}
Applied in FY26 but issued after 30 Sep 2026: 291 permits (excluded)


**What this shows**

- About 2,770 new permits applied for on or after 1 October 2025 were issued in FY26. They have about 12,000 rows, because one permit can list many locations (up to 75).
- Every permit-level column (dates, status, type, fees, review process) is the same on all of a permit's rows.
- Most of these permits still have the status "Issued"; the rest moved on to "Permit Expired", "Suspended" or "Revoked" after being issued.
- Some permits applied for in FY26 were only issued after 30 September 2026. They're outside FY26 and left out.

**Decision:** the analysis uses **one row per permit** (`TrackingNumber`). Counting rows would weight a permit with 75 locations 75 times.

## 3. What does "approve" mean?

The brief asks for days to **approve** (and once says "close"). The layer has several dates that could mark it, plus DDOT's own day counts (`DaysToApprove`, `DaysToIssue`). From here on I use one row per permit.

In [5]:
# One row per permit
fy26_permits = fy26.drop_duplicates("TrackingNumber").copy()

def days_between(start, end):
    # Calendar days between two date columns (DC dates, ignoring the time of day)
    return (fy26_permits[end].dt.normalize() - fy26_permits[start].dt.normalize()).dt.days

# The order of the stages: median calendar days between consecutive dates
stages = [("ApplicationDate", "ReadyForReviewDate"), ("ReadyForReviewDate", "LastApprovalDate"),
          ("LastApprovalDate", "PermitFeesPaymentDate"), ("PermitFeesPaymentDate", "IssueDate")]
pd.DataFrame([{"from": a, "to": b, "median days": days_between(a, b).median(), "mean days": round(days_between(a, b).mean(), 1)}
              for a, b in stages])

,from,to,median days,mean days
0,ApplicationDate,ReadyForReviewDate,0.0,0.6
1,ReadyForReviewDate,LastApprovalDate,20.0,33.7
2,LastApprovalDate,PermitFeesPaymentDate,5.0,26.8
3,PermitFeesPaymentDate,IssueDate,0.0,0.1


In [6]:
# Issue date vs fee payment: is "issued" simply the day the applicant paid?
same_moment = (fy26_permits["IssueDate"] - fy26_permits["PermitFeesPaymentDate"]).abs() < pd.Timedelta(hours=1)
print(f"Issued within an hour of paying the fees: {same_moment.mean():.1%}")

# Which dates does DDOT's own DaysToApprove count between?
for start in ["ApplicationDate", "ReadyForReviewDate"]:
    gap = fy26_permits["DaysToApprove"] - days_between(start, "LastApprovalDate")
    print(f"DaysToApprove vs {start} → LastApprovalDate: exact {(gap == 0).mean():.1%}, within a day {(gap.abs() <= 1).mean():.1%}")

Issued within an hour of paying the fees: 97.5%
DaysToApprove vs ApplicationDate → LastApprovalDate: exact 82.0%, within a day 94.2%
DaysToApprove vs ReadyForReviewDate → LastApprovalDate: exact 86.1%, within a day 99.6%


**What this shows**

The permit goes through three stages:

```
Applied ──► ready for review ──► DDOT reviews ──► APPROVED ──► applicant pays ──► ISSUED
                                                  (LastApprovalDate)               (IssueDate)
```

- The status list includes "Approved (Pending Payment)", so approval comes before payment.
- The issue date is the fee payment date (within an hour for almost all permits). The time from approval to issue is the applicant paying, sometimes months later.
- DDOT's own `DaysToApprove` counts calendar days from `ReadyForReviewDate` to `LastApprovalDate` (within a day for over 99% of permits). For most permits the application is ready for review the same day it is submitted; for about 6% it takes longer, probably because the application was incomplete.

**Decision:** approval time = **business days from `ApplicationDate` to `LastApprovalDate`**. It follows the brief's wording (application date) and measures until DDOT approves, not until the applicant pays. `IssueDate` is still used for the "issued in FY26" filter. Permits with no approval date are left out. The extra time before an application is ready for review, and the payment wait after approval, are shown separately as context.

## 4. Is FY26 complete in this layer?

Nothing in layer 23 expires before September 2026, which suggests the layer drops permits once they have expired. If so, short permits from early FY26 are gone, and the ones left are a biased sample.

DC also publishes yearly construction permit layers that keep every permit of the year. FY26 runs from October 2025 to September 2026, so the 2025 and 2026 layers together should hold all of it. I compare them with layer 23 by tracking number.

In [7]:
# Earliest expiration date in layer 23
print("Earliest expiration in layer 23:", permits["ExpirationDate"].min().date())

YEARLY = {
    2025: "https://maps2.dcgis.dc.gov/dcgis/rest/services/FEEDS/DDOT/FeatureServer/46",
    2026: "https://maps2.dcgis.dc.gov/dcgis/rest/services/FEEDS/DDOT/FeatureServer/48",
}

def download_yearly(layer_url):
    # The yearly layers, from FY26 applications on, with the columns needed here
    rows, offset = [], 0
    while True:
        params = {
            "where": "APPLICATIONDATE >= timestamp '2025-10-01 00:00:00'",
            "outFields": "TRACKINGNUMBER,PERMITNUMBER,APPLICATIONDATE,ISSUEDATE,EFFECTIVEDATE,EXPIRATIONDATE,STATUS,"
                         "ISEXCAVATION,ISFIXTURE,ISPAVING,ISLANDSCAPING,ISPROJECTIONS,ISPSRENTAL",
            "returnGeometry": "false",
            "orderByFields": "OBJECTID",
            "resultOffset": offset,
            "resultRecordCount": 2000,
            "f": "json",
        }
        page = requests.get(f"{layer_url}/query", params=params, timeout=120).json()
        if "error" in page:
            raise RuntimeError(page["error"])
        rows += [feature["attributes"] for feature in page["features"]]
        if not page.get("exceededTransferLimit"):
            break
        offset += len(page["features"])
    table = pd.DataFrame(rows)
    for column in ["APPLICATIONDATE", "ISSUEDATE", "EFFECTIVEDATE", "EXPIRATIONDATE"]:
        table[column] = pd.to_datetime(table[column], unit="ms", utc=True).dt.tz_convert(TIMEZONE)
    return table

# A permit can appear in both years, so keep one row per tracking number
yearly = pd.concat([download_yearly(url) for url in YEARLY.values()]).drop_duplicates("TRACKINGNUMBER").reset_index(drop=True)
print(f"Yearly layers: {len(yearly):,} permits applied for since 1 Oct 2025")

Earliest expiration in layer 23: 2026-09-01


Yearly layers: 24,198 permits applied for since 1 Oct 2025


In [8]:
# The yearly layers have no Renewal column. Renewal permit numbers end in "-R1", "-R2"...
# How well does that rule match layer 23's Renewal column?
renewal_in_23 = dict(zip(permits["TrackingNumber"].astype(str), permits["Renewal"]))
yearly["Renewal_in_23"] = yearly["TRACKINGNUMBER"].astype(str).map(renewal_in_23)
yearly["looks_new"] = yearly["PERMITNUMBER"].notna() & ~yearly["PERMITNUMBER"].str.contains(r"-R\d*$", na=False)
pd.crosstab(yearly["looks_new"], yearly["Renewal_in_23"].map({1: "New", 2: "Renewal"}), dropna=False)

Renewal_in_23,New,Renewal,NaN
looks_new,,,
False,1827,4597,10070
True,3080,0,4624


In [9]:
# New permits applied for and issued in FY26: all of them (yearly layers) vs those still in layer 23
yearly_fy26 = yearly[yearly["looks_new"] & (yearly["ISSUEDATE"] >= FY26_START) & (yearly["ISSUEDATE"] < FY27_START)].copy()
yearly_fy26["in_layer_23"] = yearly_fy26["TRACKINGNUMBER"].astype(str).isin(fy26_permits["TrackingNumber"].astype(str))
yearly_fy26["applied_month"] = yearly_fy26["APPLICATIONDATE"].dt.tz_localize(None).dt.to_period("M")
yearly_fy26["days_to_issue"] = (yearly_fy26["ISSUEDATE"].dt.normalize() - yearly_fy26["APPLICATIONDATE"].dt.normalize()).dt.days
yearly_fy26["permit_length"] = (yearly_fy26["EXPIRATIONDATE"].dt.normalize() - yearly_fy26["EFFECTIVEDATE"].dt.normalize()).dt.days

by_month = yearly_fy26.groupby("applied_month").agg(
    all_permits=("TRACKINGNUMBER", "size"),
    in_layer_23=("in_layer_23", "sum"),
    median_days_to_issue_all=("days_to_issue", "median"),
)
by_month["coverage"] = (by_month["in_layer_23"] / by_month["all_permits"]).round(2)
still_there = yearly_fy26[yearly_fy26["in_layer_23"]].groupby("applied_month")
by_month["median_days_to_issue_in_23"] = still_there["days_to_issue"].median()
by_month["median_permit_length_in_23"] = still_there["permit_length"].median()
print(by_month.to_string())
print(f"\nFY26 total: {len(yearly_fy26):,} new permits, {yearly_fy26['in_layer_23'].sum():,} still in layer 23 "
      f"({yearly_fy26['in_layer_23'].mean():.0%})")

               all_permits  in_layer_23  median_days_to_issue_all  coverage  median_days_to_issue_in_23  median_permit_length_in_23
applied_month                                                                                                                      
2025-10                727          150                      75.0      0.21                       276.5                       338.0
2025-11                402           83                      33.0      0.21                       266.0                       319.0
2025-12                459           34                      23.0      0.07                       159.0                       179.0
2026-01                889           24                      39.0      0.03                       106.5                       179.0
2026-02                670           89                      14.0      0.13                        84.0                       179.0
2026-03                712          141                      18.0      0.20 

**What this shows**

- Layer 23 has lost most of FY26. Only about 2,800 of roughly 7,400 new FY26 permits are still in it. For applications from October 2025 to April 2026 only 3–21% remain; from July 2026 on, 95–100%.
- The permits left from the early months are not typical: they are long permits (half a year or more) that haven't expired yet, and they took several times longer from application to issue than the early-FY26 permits as a whole.
- The permit number rule works: among FY26 applications that have a permit number and are also in layer 23, every permit without "-R" is New and every permit with "-R" is a Renewal, with no exceptions. (Permits without a permit number were never issued, so they don't matter here.)

**Decision:** an FY26 average of approval time from layer 23 would be biased high, so the analysis has two parts.

1. **Days to approve** (the brief's measure, application → `LastApprovalDate`, business days) from layer 23, on applications from **July to September 2026**, the months it still holds completely.
2. **Days to issue** (application → issue, which adds the applicant's payment time) for **all of FY26** from the yearly layers, as the full-year picture.

The completeness by month is shown in the workbook too.

### Two more checks before using the two sources

**The year-end cut-off.** The brief counts permits issued by 30 September 2026. An application from October 2025 had a whole year to be issued and still count, but one from September 2026 only counts if it was issued within days. So late-year applications that were slower are left out. How big is that effect on the July–September window?

In [10]:
# New permits applied for July–September 2026: what happened to them by 30 September?
recent = permits[(permits["Renewal"] == 1) & (permits["ApplicationDate"] >= pd.Timestamp("2026-07-01", tz=TIMEZONE))
                 & (permits["ApplicationDate"] < FY27_START)].drop_duplicates("TrackingNumber").copy()
recent["outcome"] = np.select(
    [recent["IssueDate"] < FY27_START, recent["IssueDate"] >= FY27_START, recent["LastApprovalDate"].notna()],
    ["issued by 30 Sep (counted)", "issued after 30 Sep", "approved, not issued yet"],
    "not approved yet",
)
approved = recent[recent["LastApprovalDate"].notna()].copy()
approved["days_to_approve"] = np.busday_count(approved["ApplicationDate"].dt.date.values.astype("datetime64[D]"),
                                              approved["LastApprovalDate"].dt.date.values.astype("datetime64[D]"))
print(recent["outcome"].value_counts().to_string())
print()
print(approved.groupby("outcome")["days_to_approve"].agg(permits="size", average="mean", median="median").round(1).to_string())
by_30_sep = approved[approved["LastApprovalDate"] < FY27_START]["days_to_approve"]
print(f"\nAll July–September applications approved by 30 Sep, issued or not: average {by_30_sep.mean():.1f}, median {by_30_sep.median():.0f}")

outcome
issued by 30 Sep (counted)    1229
not approved yet               545
approved, not issued yet       341
issued after 30 Sep            167

                            permits  average  median
outcome                                             
approved, not issued yet        341     19.6    17.0
issued after 30 Sep             167     16.3    13.0
issued by 30 Sep (counted)     1227      8.9     2.0

All July–September applications approved by 30 Sep, issued or not: average 10.5, median 8


**The yearly layers' time stamps.** For the same permit, the yearly layers' dates are exactly 4 hours earlier than layer 23's (5 hours in winter): they store DC local time but label it as UTC. Read as DC local time, they match.

In [11]:
# Compare the same permits' application dates in both sources
both = fy26_permits.assign(tn=fy26_permits["TrackingNumber"].astype(str)).merge(
    yearly.assign(tn=yearly["TRACKINGNUMBER"].astype(str)), on="tn")
shift_hours = (both["APPLICATIONDATE"] - both["ApplicationDate"]).dt.total_seconds() / 3600
print("Hours between the two sources' application dates:", shift_hours.round(1).value_counts().to_dict())
print("Status the same in both:", (both["StatusDescription"] == both["STATUS"]).mean())

Hours between the two sources' application dates: {-4.0: 2753, -5.0: 13}
Status the same in both: 1.0


**What this shows**

- Among July–September applications, the ones issued after 30 September or not yet paid for took about twice as long to approve as the ones that count. The brief's "issued in FY26" rule therefore makes recent approval times look a little faster. I report the brief's figure and mention the fairer one (all applications approved by 30 September, issued or not) alongside it.
- The yearly layers agree with layer 23 on every permit they share once their times are read as DC local time. The preparation script reads them that way.

## 5. Permit types

The brief asks for approval time by permit type category, and leaves the categories to me. The layer describes the work in two ways: yes/no flags (`IsExcavation`, `IsPaving`, …) and a detailed list of work items (`TypeDetailNames`, e.g. "Excavation; Repair Gas Main Line"). The yearly layers only have six of the flags, so categories built from those six can be used in both parts of the analysis.

Here I look at the permits applied for from July to September 2026 (complete in this layer). Durations skip weekends only; DC holidays are added in the data preparation step.

In [12]:
JULY_2026 = pd.Timestamp("2026-07-01", tz=TIMEZONE)
complete = fy26_permits[(fy26_permits["ApplicationDate"] >= JULY_2026) & fy26_permits["LastApprovalDate"].notna()].copy()

# Business days from application to approval (weekends skipped; holidays come later)
complete["days_to_approve"] = np.busday_count(
    complete["ApplicationDate"].dt.date.values.astype("datetime64[D]"),
    complete["LastApprovalDate"].dt.date.values.astype("datetime64[D]"),
)
print(f"Permits applied for July–September 2026: {len(complete):,}")

# How often is each flag set, and how often are several set at once?
flags = ["IsExcavation", "IsPaving", "IsFixture", "IsLandscaping", "IsProjections", "IsPSRental", "IsOverHeadWork"]
print(complete[flags].sum().to_string())
print("Permits with more than one flag:", (complete[flags].sum(axis=1) > 1).sum())

Permits applied for July–September 2026: 1,228
IsExcavation      1112
IsPaving            38
IsFixture           42
IsLandscaping        5
IsProjections       11
IsPSRental           0
IsOverHeadWork      52
Permits with more than one flag: 27


In [13]:
def work_type(row):
    # Level 1: from the flags shared with the yearly layers (overhead work is in layer 23 only)
    types = [name for flag, name in [("IsExcavation", "Excavation"), ("IsPaving", "Paving"), ("IsFixture", "Fixture"),
                                     ("IsOverHeadWork", "Overhead work")] if row[flag] == 1]
    if len(types) > 1:
        return "Multiple types"
    if types:
        return types[0]
    return "Other (projections, landscaping, rental)"

def excavation_detail(work_items):
    # Level 2: which utility the excavation is for, from the detailed work items
    text = str(work_items).lower()
    found = [name for name, words in [("Gas", "gas"), ("Water", "water|hydrant|valve"), ("Sewer", "sewer|clean out"),
                                      ("Electric", "electric|conduit|cathodic"), ("Telecom", "telecom|fiber|cable"),
                                      ("Test pits", "boring|test pit")] if re.search(words, text)]
    if len(found) > 1:
        return "Several utilities"
    return found[0] if found else "Other excavation"

complete["work_type"] = complete.apply(work_type, axis=1)
complete["excavation_for"] = np.where(complete["work_type"] == "Excavation",
                                      complete["TypeDetailNames"].map(excavation_detail), None)

summary = lambda column: (complete.groupby(column)["days_to_approve"]
                          .agg(permits="size", average="mean", median="median").round(1)
                          .sort_values("permits", ascending=False))
print(summary("work_type").to_string())
print()
print(summary("excavation_for").to_string())

                                          permits  average  median
work_type                                                         
Excavation                                   1097      7.6     1.0
Overhead work                                  38     23.0    22.0
Fixture                                        36     12.6     9.0
Paving                                         33     23.6    23.0
Multiple types                                 18     21.6    17.5
Other (projections, landscaping, rental)        6     15.5    14.5

                   permits  average  median
excavation_for                             
Gas                    483      5.8     1.0
Water                  244      3.7     1.0
Sewer                  111      1.0     0.0
Electric               107     15.5    14.0
Several utilities       85     25.3    22.0
Other excavation        36      5.3     1.0
Test pits               25     18.0    16.0
Telecom                  6     16.5    15.0


**What this shows**

- Nine in ten new permits are excavation, so a split by flag alone hides most of the variation.
- Inside excavation the differences are large. Routine sewer, water and gas work is usually approved within about a day (median 0–1 business days). Electric work, test pits and permits covering several utilities take two to four weeks.
- Paving, fixture and overhead work permits also take two to four weeks.

**Decision:** two levels of category.

1. **Work type** (from the flags): Excavation, Paving, Fixture, Overhead work, Multiple types, Other. Used in both parts of the analysis. The yearly layers have no overhead-work flag, so in the full-year part those permits fall under Other.
2. **Excavation for** (from the work items): Gas, Water, Sewer, Electric, Telecom, Test pits, Several utilities, Other excavation. Approval part only, as a drill-down.

## 6. Another helpful split

The brief asks for one more segment that shows where approval times are meaningfully different. Candidates: who applies, whether the applicant had to revise and resubmit, how many locations the permit covers, and the ward. Same July–September 2026 permits as section 5.

In [14]:
def applicant(row):
    # Who the permit is for, from the permittee, applicant and owner names
    names = " ".join(str(row[column]) for column in ["PermitteeName", "ApplicantCompany", "OwnerName"]).lower()
    for group, pattern in [("Washington Gas", r"wash gas|washington gas|wgl"), ("DC Water", r"dc ?water|wasa"),
                           ("PEPCO", r"pepco|potomac electric"),
                           ("Telecom", r"verizon|comcast|zayo|crown castle|at&t|lumen|cogent|rcn"),
                           ("DC government", r"department|district of columbia|ddot|doee|public schools|office of")]:
        if re.search(pattern, names):
            return group
    return "Private / other"

locations_per_permit = permits.groupby("TrackingNumber").size()

complete["applicant"] = complete.apply(applicant, axis=1)
complete["resubmission_requested"] = complete["LastResubmissionRequestDate"].notna().map({True: "Yes", False: "No"})
complete["locations"] = pd.cut(complete["TrackingNumber"].map(locations_per_permit), [0, 1, 3, 10, 1000],
                               labels=["1", "2–3", "4–10", "11 or more"])

def split(column):
    # Business days to approve for each group, and the share approved the same day
    return (complete.groupby(column, observed=True)["days_to_approve"]
            .agg(permits="size", average="mean", median="median", same_day=lambda days: (days == 0).mean())
            .round(2).sort_values("permits", ascending=False))

for column in ["applicant", "resubmission_requested", "locations", "WorkLocationWard"]:
    print(split(column).to_string(), "\n")

                 permits  average  median  same_day
applicant                                          
Washington Gas       540     6.39     1.0      0.38
DC Water             439     6.21     1.0      0.41
PEPCO                108    20.00    16.0      0.00
Private / other       95    18.95    18.0      0.02
Telecom               38    17.58    16.0      0.03
DC government          8    15.62    13.5      0.00 

                        permits  average  median  same_day
resubmission_requested                                    
No                         1050     6.34     1.0      0.37
Yes                         178    23.99    22.0      0.01 

            permits  average  median  same_day
locations                                     
1              1112     7.35     1.0      0.35
2–3              60    19.28    16.5      0.00
4–10             34    24.76    24.0      0.00
11 or more       22    34.36    30.0      0.00 

                  permits  average  median  same_day
WorkLoc

**What this shows**

- **Resubmission** makes the biggest single difference: permits that were sent back for revision took about 24 business days on average (median 22), against about 6 (median 1) for permits approved without one.
- **Who applies:** Washington Gas and DC Water permits are usually approved within a day; PEPCO, telecom and private applicants take three to four weeks. This partly repeats the permit type split (gas and water repairs are the fast ones).
- **Locations:** approval time rises steadily with the number of locations, from a median of 1 day for one location to 30 for eleven or more.
- **Ward:** no clear pattern.

**Decision:** sheet 3 is **resubmission requested (yes/no)**. It's the clearest split, and it points to a next step: why applications are sent back, and whether better guidance could avoid it. Applicant and number of locations go on extra sheets.